# Crutch curriculum v4 — one episode per stage

Runs a single episode of stages A, B, C and D, then plots the rewards.

There is **no trained v4 checkpoint** (the observation layout changed when
`prev_action` was added), so these are diagnostics of the environment and the
reward function, not evaluations of a policy. The default policy emits zero
actions; the rate limiter means the applied torque stays at zero throughout, so
what you are looking at is the reward the model collects while falling under
gravity from its initialised pose.

Requires `sconegym` and a licensed Hyfydy (`sconepy`). Run the first cell before
anything imports numpy.

## 1. CPU budget

Half the logical cores. Importing `run_stages` sets the BLAS thread
variables before it imports numpy, so keep this as the first cell you run.

In [ ]:
import sys
from pathlib import Path

V4 = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("v4")
sys.path[:0] = [str(V4), str(V4 / "scripts")]

import run_stages

cpu = run_stages.set_cpu_budget(fraction=0.5)

`psutil` gives a hard affinity cap. Without it only the cooperative thread-count
variables are set, which is usually enough here because the Hyfydy integrator is
serial — but install it if you want the guarantee:

```
pip install psutil
```

## 2. What the stages are

In [ ]:
import sconegym_crutch_v4 as scv4

for key in scv4.STAGE_ORDER:
    stage = scv4.STAGES[key]
    print(scv4.ENV_IDS[key])
    print("   ", stage.describe())
    print("    crutch: force={} pose={} | target_vel={:.3f} | fall_penalty={:.1f}".format(
        stage.needs_crutch_force, stage.needs_crutch_pose,
        stage.target_vel, stage.reward.fall_penalty))

## 3. Run one episode per stage

`strict_crutch=True` makes stages B, C and D refuse to construct if the crutch
contact-force API is not working. That is deliberate — a silent zero there is
what made the v3 crutch reward meaningless. If a stage fails here, that is the
finding; set `strict_crutch=False` to proceed anyway and read the force trace in
section 5 to see what the API actually returns.

In [ ]:
results = run_stages.run_all(
    stages=("A", "B", "C", "D"),
    policy="zeros",      # "random", or "checkpoint:/path/to/step_XXXXXXX"
    seed=0,
    strict_crutch=True,
)

In [ ]:
print(run_stages.summarise(results))

## 4. Rewards

In [ ]:
%matplotlib inline
fig = run_stages.plot_rewards(results, save_dir=V4 / "notebooks" / "figures")

Expect the per-step reward to start near the stage maximum and decay as the
model loses posture, with the episode ending early on a fall. The dashed line at
zero is the invariant worth checking: **no step should be negative.** Every term
is in `[0, 1]` and penalties are gates rather than subtractions, so a negative
step reward means something is wrong with the composition, not with the policy.

## 5. Individual reward terms

In [ ]:
fig = run_stages.plot_terms(results, save_dir=V4 / "notebooks" / "figures")

In [ ]:
for key, res in results.items():
    if res.get("ok") and res.get("crutch_force"):
        import numpy as np
        arr = np.asarray(res["crutch_force"])
        print("stage %s crutch force (N): min %.2f mean %.2f max %.2f"
              % (key, arr.min(), arr.mean(), arr.max()))
        if arr.max() <= 0.0:
            print("   ^ zero for the whole episode: the crutch term is measuring nothing")

## 6. Summary

In [ ]:
fig = run_stages.plot_summary(results, save_dir=V4 / "notebooks" / "figures")
print("saved:", run_stages.save_results(results))

## 7. Reward gating — runs without a simulator

This section drives `RewardSpec.compose` directly, so it works on any machine
with numpy and matplotlib. Every term is held at 1.0 while one is swept down to
0.0, showing how much reward a policy keeps by abandoning a single objective.

A flat line is a term the policy can safely ignore. A steep one is a term that
genuinely gates. Watch the trend across stages: **gating power dilutes as the
term count grows**, because each term's normalised exponent shrinks. Stage A
punishes abandoning `posture` down to 0.32; stage D leaves 0.60 for the same
choice, and abandoning `displacement` there costs almost nothing.

If stage D will not walk, this is the plot that says the fix is fewer terms or a
lower `term_floor` — not bigger weights.

In [ ]:
fig = run_stages.plot_gating(save_dir=V4 / "notebooks" / "figures")

## Troubleshooting

| symptom | meaning |
|---|---|
| `ModuleNotFoundError: sconepy` | Hyfydy is not installed or not licensed on this machine. |
| stage B/C/D `construction failed` | The crutch contact-force probe failed. Read the message; this is the single most important thing to resolve. |
| a negative step reward | A composition bug. `python -m pytest ../tests -q` should catch it. |
| all four stages end in a few steps | Expected with the zero-action policy — no torque is applied, so the model simply falls. |